# 12 — Синтез датасета через back-translation (SQL → NL)

> **Проблема:** валидационный датасет «NL → SQL → vuln_class» не
> поставляется. Без него нечем мерить EX и Recall судьи.
>
> **Решение (ADR-0006):** берём SQL (свои + адаптации PortSwigger/sqlmap),
> просим LLM сгенерировать **NL-формулировку** к каждому. NL генерируется
> проще, чем SQL — это позволяет за пару долларов собрать домен-специфичный
> eval-set.

## Что покажем

1. Seed pool из 10 SQL (8 безопасных + 2 уязвимых).
2. Mock-LLM `sql_to_text`: словарь шаблонов под SQL-паттерны.
3. Валидация: pglast-парсинг, sandbox-исполнение, sanity check.
4. Quality-gate: «Phase 1 должен подтвердить vuln_class».
5. Train/eval split со стратификацией.


## 🧒 Аналогия для ребёнка

Учительница пишет **ответ задачи** (например, «42»). А потом
нужно **придумать сам вопрос** так, чтобы ответ был 42:
«сколько будет 6×7?», «сколько лет в полувеке?» и т.д.

Это и есть **back-translation**: у нас есть готовый SQL (ответ),
и LLM придумывает к нему вопрос на естественном языке.

Это легче, чем обратное — генерировать SQL по вопросу (там надо
знать схему и быть точным; вопросы же — гибкий язык).


## 1. SQL seed pool


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


##
# @brief Seed pool — 10 SQL, рукописных под нашу мок-схему.
SEED_SQL = [
    # safe (8)
    ("SELECT id, full_name, balance FROM clients WHERE balance > 0 ORDER BY balance DESC LIMIT 100",   "safe", "easy"),
    ("SELECT COUNT(*) FROM credit_contract WHERE status_id = 3",                                       "safe", "easy"),
    ("SELECT c.full_name, SUM(p.amount) FROM clients c JOIN payment p ON p.contract_id IN (SELECT id FROM credit_contract WHERE client_id=c.client_id) GROUP BY c.full_name LIMIT 50", "safe", "medium"),
    ("SELECT contract_id, MAX(date) FROM payment GROUP BY contract_id LIMIT 200",                      "safe", "medium"),
    ("SELECT segment_name, COUNT(*) FROM business_segment GROUP BY segment_name",                       "safe", "easy"),
    ("SELECT a.account_name, t.amount FROM acc_number a JOIN transaction_log t ON t.account_id=a.id WHERE t.ts > '2026-01-01' LIMIT 1000", "safe", "medium"),
    ("SELECT name FROM dict_product WHERE type='credit'",                                              "safe", "easy"),
    ("SELECT contract_id, amount FROM payment WHERE date BETWEEN '2026-01-01' AND '2026-04-01' LIMIT 500", "safe", "easy"),
    # vulnerable (2)
    ("DELETE FROM credit_contract",                                                                   "DML_NO_WHERE", "easy"),
    ("SELECT * FROM clients",                                                                         "SELECT_STAR", "easy"),
]

print(f"Seed pool: {len(SEED_SQL)} SQL")
for sql, vc, diff in SEED_SQL[:3]:
    print(f"  [{vc:14s}] [{diff:6s}] {sql[:60]}...")


## 2. Mock-LLM: SQL → NL

В реальной системе тут GPT-4o-mini или Qwen-Coder с промптом
«опиши пошагово что делает SQL → сформулируй 2 NL-вопроса».
Здесь — pattern-based mock на ключевых словах SQL.


In [ ]:
##
# @brief Mock-LLM: SQL → 2 NL-вопроса (short, long).
def mock_sql_to_text(sql):
    sql_lo = sql.lower()
    if "delete from" in sql_lo:
        m = re.search(r"delete from (\w+)", sql_lo)
        tbl = m.group(1) if m else "таблицы"
        return {
            "nl_short": f"очисти таблицу {tbl}",
            "nl_long":  f"мне нужно полностью удалить все записи из {tbl}, это будет очистка",
        }
    if "count(*)" in sql_lo:
        m = re.search(r"from (\w+)", sql_lo)
        tbl = m.group(1) if m else "таблицы"
        return {
            "nl_short": f"сколько строк в {tbl}",
            "nl_long":  f"посчитай, пожалуйста, сколько всего записей в {tbl} с учётом фильтра",
        }
    if "join" in sql_lo:
        return {
            "nl_short": "связь данных из нескольких таблиц",
            "nl_long":  "нужен отчёт, объединяющий информацию из разных таблиц по ключу",
        }
    if "select *" in sql_lo:
        m = re.search(r"from (\w+)", sql_lo)
        tbl = m.group(1) if m else "таблицы"
        return {
            "nl_short": f"выгрузи всё из {tbl}",
            "nl_long":  f"экспортируй полностью все данные из таблицы {tbl} в csv",
        }
    if "group by" in sql_lo:
        return {
            "nl_short": "сгруппируй и подсчитай",
            "nl_long":  "сделай агрегацию по группам с подсчётом сумм/количеств",
        }
    return {
        "nl_short": "сделай выборку",
        "nl_long":  "вытащи данные согласно условиям",
    }


section("Тест back-translation на 3 примерах")
for sql, vc, diff in SEED_SQL[:3]:
    nl = mock_sql_to_text(sql)
    print(f"\n  SQL:      {sql[:60]}")
    print(f"  vuln:     {vc}")
    print(f"  nl_short: {nl['nl_short']}")
    print(f"  nl_long:  {nl['nl_long']}")


## 3. Валидация — pglast + sandbox-исполнение


In [ ]:
##
# @brief Sandbox: имитация Postgres через sqlite3 для проверки исполнимости SELECT.
def setup_sandbox():
    conn = sqlite3.connect(":memory:")
    # Минимальные таблицы для seed pool
    for ddl in [
        "CREATE TABLE clients (client_id INT, id INT, full_name TEXT, balance REAL)",
        "CREATE TABLE credit_contract (id INT, client_id INT, status_id INT, amount REAL)",
        "CREATE TABLE payment (id INT, contract_id INT, amount REAL, date TEXT)",
        "CREATE TABLE business_segment (id INT, client_id INT, segment_name TEXT)",
        "CREATE TABLE acc_number (id INT, account_name TEXT)",
        "CREATE TABLE transaction_log (id INT, account_id INT, amount REAL, ts TEXT)",
        "CREATE TABLE dict_product (id INT, name TEXT, type TEXT)",
    ]:
        conn.execute(ddl)
    # Сидинг 5 строк на таблицу
    for _ in range(5):
        conn.execute("INSERT INTO clients VALUES (?, ?, ?, ?)", (1, 1, "Иван", 100.0))
        conn.execute("INSERT INTO credit_contract VALUES (?, ?, ?, ?)", (1, 1, 3, 50000))
        conn.execute("INSERT INTO payment VALUES (?, ?, ?, ?)", (1, 1, 1000, "2026-02-15"))
        conn.execute("INSERT INTO business_segment VALUES (?, ?, ?)", (1, 1, "retail"))
        conn.execute("INSERT INTO acc_number VALUES (?, ?)", (1, "Расчётный"))
        conn.execute("INSERT INTO transaction_log VALUES (?, ?, ?, ?)", (1, 1, 50, "2026-02-15"))
        conn.execute("INSERT INTO dict_product VALUES (?, ?, ?)", (1, "Ипотека", "credit"))
    conn.commit()
    return conn


def validate(sql, conn):
    """@brief Возвращает (ok, error_msg)."""
    # 1. Парсинг (в проде — pglast, тут — sqlite EXPLAIN)
    try:
        conn.execute(f"EXPLAIN {sql}")
    except sqlite3.Error as e:
        return False, f"parse error: {e}"
    # 2. Для SELECT — попробовать выполнить
    if sql.strip().upper().startswith("SELECT"):
        try:
            conn.execute(sql).fetchall()
        except sqlite3.Error as e:
            return False, f"runtime error: {e}"
    return True, ""


sandbox = setup_sandbox()
section("Валидация всех seed SQL")
for sql, vc, diff in SEED_SQL:
    ok, err = validate(sql, sandbox)
    status = "✅" if ok else "❌"
    print(f"  {status} [{vc:14s}] {sql[:50]}...  {err}")


## 4. Quality-gate: Phase 1 должен подтвердить vuln_class


In [ ]:
def quality_check_vuln(sql, expected_vuln_class):
    """@brief Проверяет, что Phase 1 правила подтверждают заявленный класс."""
    if expected_vuln_class == "safe":
        # Не должно быть ни одного finding (упрощённо)
        return True
    # SELECT * → SELECT_STAR
    if expected_vuln_class == "SELECT_STAR":
        return bool(re.search(r"SELECT\s+\*", sql, re.IGNORECASE))
    if expected_vuln_class == "DML_NO_WHERE":
        return bool(re.search(r"^\s*(DELETE|UPDATE)\b(?!.*WHERE)", sql,
                              re.IGNORECASE | re.DOTALL))
    return False


section("Quality-gate: vuln_class согласован с правилами?")
for sql, vc, diff in SEED_SQL:
    ok = quality_check_vuln(sql, vc)
    print(f"  {'✅' if ok else '⚠️ '} [{vc:14s}] {sql[:50]}")


## 5. Train/eval split со стратификацией


In [ ]:
import random


def stratified_split(samples, eval_ratio=0.2, seed=42):
    """@brief Разбиваем по vuln_class сбалансированно."""
    rng = random.Random(seed)
    by_class = {}
    for s in samples:
        by_class.setdefault(s[1], []).append(s)
    train, eval_ = [], []
    for vc, items in by_class.items():
        rng.shuffle(items)
        n_eval = max(1, int(len(items) * eval_ratio))
        eval_.extend(items[:n_eval])
        train.extend(items[n_eval:])
    return train, eval_


train, eval_set = stratified_split(SEED_SQL)
section("Split со стратификацией")
print(f"  train: {len(train)} примеров, eval: {len(eval_set)} примеров")
print(f"  по классам в eval:")
counts = {}
for s in eval_set:
    counts[s[1]] = counts.get(s[1], 0) + 1
for vc, n in counts.items():
    print(f"    {vc}: {n}")


## Итог

Мы увидели проблему **под микроскопом** и **симуляцию решения** из ADR.

## Куда дальше

- **Описание проблемы:** [problems/engineering/03-synthetic-dataset/README.md](../problems/engineering/03-synthetic-dataset/README.md)
- **Варианты решения + почему так:** [problems/engineering/03-synthetic-dataset/solutions.md](../problems/engineering/03-synthetic-dataset/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../docs/adr/0002-loop-architecture-langgraph.md)
